# Retrieving GDC Pathology Reports for TCGA Cases in IDC

TCGA whole-slide images in IDC are linked to pathology report PDFs hosted on the NCI Genomic Data Commons (GDC).  
The join key is the **TCGA case barcode** (`PatientID` in IDC, `submitter_id` in GDC).

**Coverage (TCGA-BRCA baseline):** 1,094 of 1,098 IDC patients (99.6%) have a pathology report on GDC.  
A small number of patients have two reports; the rest have exactly one.

**No authentication required** — GDC pathology reports are open-access files.

## Requirements

```
pip install idc-index requests pymupdf
```

In [ ]:
import csv
import io
import json
import re
import time
from collections import Counter, defaultdict
from pathlib import Path

import fitz          # pymupdf
import pandas as pd
import requests
from idc_index import IDCClient

GDC_FILES_URL = "https://api.gdc.cancer.gov/files"
GDC_DATA_URL  = "https://api.gdc.cancer.gov/data"

client = IDCClient()
print("IDC version:", client.get_idc_version())

## 1. Get TCGA Patient IDs from IDC

`PatientID` in IDC equals the GDC `submitter_id` for TCGA collections.

In [ ]:
COLLECTION = "tcga_brca"   # change to any tcga_* collection_id

df = client.sql_query(f"""
    SELECT DISTINCT PatientID
    FROM index
    WHERE collection_id = '{COLLECTION}'
    ORDER BY PatientID
""")

patient_ids = list(df["PatientID"])
print(f"{len(patient_ids)} patients in {COLLECTION}")
print("Sample:", patient_ids[:5])

## 2. Query GDC for Pathology Report File Metadata

GDC stores pathology reports as open-access PDF files.
We filter by `data_type = "Pathology Report"` and match on `cases.submitter_id`.

Batching is required — GDC filters with `in` operator accept up to ~500 values per request.

In [ ]:
def query_gdc_pathology_reports(submitter_ids, batch_size=200):
    """Return list of GDC file metadata dicts for the given TCGA case submitter_ids."""
    all_hits = []
    for i in range(0, len(submitter_ids), batch_size):
        batch = submitter_ids[i : i + batch_size]
        payload = {
            "filters": json.dumps({
                "op": "and",
                "content": [
                    {"op": "in", "content": {"field": "cases.submitter_id", "value": batch}},
                    {"op": "=",  "content": {"field": "data_type", "value": "Pathology Report"}},
                ],
            }),
            "fields": "file_id,file_name,cases.submitter_id,file_size,md5sum",
            "size": str(batch_size),
            "format": "JSON",
        }
        r = requests.get(GDC_FILES_URL, params=payload, timeout=30)
        r.raise_for_status()
        all_hits.extend(r.json()["data"]["hits"])
    return all_hits


reports = query_gdc_pathology_reports(patient_ids)
print(f"{len(reports)} reports found for {len(patient_ids)} IDC patients")

## 3. Build a Lookup Table: Case → Report(s)

Most patients have exactly one report; a few have two.

In [ ]:
case_to_reports = defaultdict(list)
for hit in reports:
    submitter_id = hit["cases"][0]["submitter_id"]
    case_to_reports[submitter_id].append(hit)

idc_patients_set = set(patient_ids)
matched   = idc_patients_set & case_to_reports.keys()
unmatched = idc_patients_set - case_to_reports.keys()

print(f"IDC patients with ≥1 GDC report : {len(matched)} ({len(matched)/len(patient_ids)*100:.1f}%)")
print(f"IDC patients with no GDC report  : {len(unmatched)}")
print("Report-count distribution:", dict(Counter(len(v) for v in case_to_reports.values())))

sample_case   = next(iter(matched))
sample_report = case_to_reports[sample_case][0]
print(f"\nExample — {sample_case}:")
print(f"  file_id  : {sample_report['file_id']}")
print(f"  file_name: {sample_report['file_name']}")
print(f"  file_size: {sample_report['file_size']:,} bytes")
print(f"  download : {GDC_DATA_URL}/{sample_report['file_id']}")

## 4. Download Reports

Files are downloaded one at a time with `GET /data/{file_id}`.  
GDC does not require authentication for open-access files.

**Bulk download** of many files is faster with the [GDC Data Transfer Tool](https://gdc.cancer.gov/access-data/gdc-data-transfer-tool) (`gdc-client`), which supports parallel downloads from a manifest. The manifest format is shown in section 5.

In [ ]:
def download_gdc_file(file_id: str, dest_path: Path, chunk_size: int = 1 << 20) -> Path:
    """Stream a GDC file to disk. Returns the destination path."""
    dest_path.parent.mkdir(parents=True, exist_ok=True)
    with requests.get(f"{GDC_DATA_URL}/{file_id}", stream=True, timeout=60) as r:
        r.raise_for_status()
        with open(dest_path, "wb") as f:
            for chunk in r.iter_content(chunk_size):
                f.write(chunk)
    return dest_path


def download_reports_for_collection(
    case_to_reports: dict,
    output_dir: str | Path,
    limit: int | None = None,
    delay_s: float = 0.2,
) -> list[Path]:
    """
    Download pathology report PDFs.

    Parameters
    ----------
    case_to_reports : dict mapping submitter_id → list of GDC file-metadata dicts
    output_dir      : destination directory; PDFs are saved as {submitter_id}/{file_name}
    limit           : max number of cases to download (None = all)
    delay_s         : polite pause between requests
    """
    output_dir = Path(output_dir)
    downloaded = []
    cases = list(case_to_reports.items())
    if limit:
        cases = cases[:limit]

    for case_id, file_list in cases:
        for meta in file_list:
            dest = output_dir / case_id / meta["file_name"]
            if dest.exists():
                print(f"  skip (exists): {dest.name}")
                downloaded.append(dest)
                continue
            print(f"  downloading {case_id} — {meta['file_name']} ({meta['file_size']:,} B)")
            download_gdc_file(meta["file_id"], dest)
            downloaded.append(dest)
            time.sleep(delay_s)

    return downloaded


# Download a small sample (3 cases)
paths = download_reports_for_collection(
    case_to_reports,
    output_dir=f"/tmp/gdc_pathology/{COLLECTION}",
    limit=3,
)
for p in paths:
    print(p, f"({p.stat().st_size:,} bytes)")

## 5. Generate a GDC Download Manifest

For large downloads the [GDC Data Transfer Tool](https://gdc.cancer.gov/access-data/gdc-data-transfer-tool) is more efficient.  
A manifest is a TSV with columns `id`, `filename`, `md5`, `size`, `state`.

In [ ]:
def make_gdc_manifest(reports: list[dict]) -> str:
    """Return manifest TSV string suitable for: gdc-client download -m manifest.txt"""
    buf = io.StringIO()
    writer = csv.writer(buf, delimiter="\t")
    writer.writerow(["id", "filename", "md5", "size", "state"])
    for r in reports:
        writer.writerow([
            r["file_id"],
            r["file_name"],
            r.get("md5sum", ""),
            r.get("file_size", ""),
            "released",
        ])
    return buf.getvalue()


manifest = make_gdc_manifest(reports)
manifest_path = Path(f"/tmp/gdc_pathology/{COLLECTION}_manifest.txt")
manifest_path.parent.mkdir(parents=True, exist_ok=True)
manifest_path.write_text(manifest)

print(f"Manifest written: {manifest_path} ({len(reports)} files)")
print("\n".join(manifest.splitlines()[:4]))
print()
print(f"# gdc-client download -m {manifest_path} -d /path/to/output")

## 6. Text Extraction

All GDC pathology report PDFs contain a **text layer** — no OCR is required. `fitz` (PyMuPDF) extracts it reliably.

### Document types

Two distinct types exist in the corpus with very different extraction quality:

| Type | How to detect | Typical size | Usable content |
|------|--------------|-------------|----------------|
| **Pathology report** | Contains diagnosis narrative | 400 – 10,000+ chars | High — structured prose with TNM, grade, margin, LVI |
| **Missing Report Form** | `'Missing Pathology Report Form'` in text | ~1,400 chars | Low — form template is clean; handwritten field values are absent or garbled |

### PDF structure notes

- Many reports embed a full-page scanned image (3,400 × 4,400 px ≈ 400 DPI letter). The text layer is either the original digital source or OCR placed over the scan.
- A **TCGA QA checklist footer** appears on almost every report (Diagnosis Discrepancy, HIPAA Discrepancy, Prior Malignancy History, Dual/Synchronous checkboxes). Checkbox values in that section are unreliable — rendered as barcode-like strings.
- Core diagnostic narrative text is clean in the vast majority of reports.

### Library comparison

All four libraries (`fitz`, `pypdf`, `pdfminer.six`, `pdfplumber`) extract equivalent content from these files. `fitz` is ~3× faster; `pdfminer.six` adds slightly cleaner inter-line whitespace. `fitz` is recommended.

In [ ]:
def extract_text(pdf_path: Path) -> str:
    """Extract full text from a PDF using PyMuPDF."""
    doc = fitz.open(pdf_path)
    text = "\n".join(page.get_text() for page in doc)
    doc.close()
    return text


def classify_report(text: str) -> str:
    """Return 'missing_form' or 'pathology_report'."""
    return "missing_form" if "Missing Pathology Report Form" in text else "pathology_report"


# Demo on the 3 downloaded reports
for p in sorted(Path(f"/tmp/gdc_pathology/{COLLECTION}").rglob("*.PDF")):
    text = extract_text(p)
    doc = fitz.open(p)
    pages  = len(doc)
    images = sum(len(pg.get_images()) for pg in doc)
    doc.close()
    print(f"{p.parent.name}  [{classify_report(text)}]  "
          f"pages={pages}  images={images}  chars={len(text):,}")
    print(text[:300].strip())
    print()

### Structured field extraction

For pathology reports the text is clean enough for regex-based extraction of key staging fields.
For production use, consider [scispaCy](https://allenai.github.io/scispacy/) or an LLM for more robust entity recognition.

In [ ]:
def extract_staging_fields(text: str) -> dict:
    """Best-effort regex extraction of common pathology staging fields."""
    fields = {}

    # TNM — pT*, pN*, pM* (including combined forms like pT2pN0)
    tnm = re.findall(r'p[TtNnMm]\d[\w]*', text)
    if tnm:
        fields["tnm"] = list(dict.fromkeys(tnm))  # deduplicated, order preserved

    # Histologic grade — G1–G4 or 'grade N' or 'SBR grade N'
    grade = re.findall(r'\bG\s*[1-4]\b|[Gg]rade\s+[1-4]|SBR\s+grade\s+\d', text)
    if grade:
        fields["grade"] = list(dict.fromkeys(grade))

    # Tumor size — e.g. '2.1 cm', '35 mm'
    sizes = re.findall(r'\b(\d+\.?\d*)\s*(cm|mm)\b', text, re.IGNORECASE)
    if sizes:
        fields["size_mentions"] = [f"{v} {u}" for v, u in sizes[:5]]

    # Lymph node summary — 'X of Y lymph nodes'
    ln = re.findall(
        r'(\d+)\s*(?:of|/)\s*(\d+)\s*(?:lymph\s*nodes?|axillary)', text, re.IGNORECASE
    )
    if ln:
        fields["lymph_nodes_pos_total"] = [f"{pos}/{total}" for pos, total in ln]

    # Margin status
    if re.search(r'margin[s]?\s*(?:are\s*)?(?:clear|negative|free|tumor.free)', text, re.IGNORECASE):
        fields["margins"] = "negative"
    elif re.search(r'margin[s]?\s*(?:are\s*)?positive|tumor\s+at\s+margin', text, re.IGNORECASE):
        fields["margins"] = "positive"

    # Vascular / lymphatic invasion
    vi = re.search(
        r'(?:vascular|lymphatic|lymphovascular)\s+invasion[:\s]+(present|not\s+present|absent)',
        text, re.IGNORECASE
    )
    if vi:
        fields["vascular_invasion"] = vi.group(1).lower()

    return fields


for p in sorted(Path(f"/tmp/gdc_pathology/{COLLECTION}").rglob("*.PDF")):
    text = extract_text(p)
    if classify_report(text) == "missing_form":
        print(f"{p.parent.name}  [missing_form — skipped]")
        continue
    fields = extract_staging_fields(text)
    print(f"{p.parent.name}")
    for k, v in fields.items():
        print(f"  {k:25s}: {v}")
    print()

## 7. Multi-Collection Survey

Check GDC report availability across all TCGA collections present in IDC.

In [ ]:
tcga_collections = client.sql_query("""
    SELECT collection_id, COUNT(DISTINCT PatientID) AS idc_patients
    FROM index
    WHERE collection_id LIKE 'tcga_%'
    GROUP BY collection_id
    ORDER BY idc_patients DESC
""")

rows = []
for _, row in tcga_collections.iterrows():
    coll_id    = row["collection_id"]
    gdc_project = coll_id.replace("tcga_", "TCGA-").upper()
    n_idc      = int(row["idc_patients"])
    payload = {
        "filters": json.dumps({"op": "and", "content": [
            {"op": "=", "content": {"field": "cases.project.project_id", "value": gdc_project}},
            {"op": "=", "content": {"field": "data_type", "value": "Pathology Report"}},
        ]}),
        "fields": "file_id", "size": "1", "format": "JSON",
    }
    r = requests.get(GDC_FILES_URL, params=payload, timeout=30)
    total_gdc = r.json()["data"]["pagination"]["total"]
    rows.append({"collection_id": coll_id, "idc_patients": n_idc, "gdc_reports": total_gdc})
    time.sleep(0.1)

survey = pd.DataFrame(rows)
survey["coverage_pct"] = (survey["gdc_reports"] / survey["idc_patients"] * 100).round(1)
print(survey.to_string(index=False))